In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
mnist = './Data/mnist.npz'


# Load the data
with np.load(mnist) as data:
    x_train = data['x_train']
    y_train = data['y_train']
    x_test = data['x_test']
    y_test = data['y_test']


x_train = x_train.reshape(-1,28*28).astype('float32' )/ 255.0
x_test = x_test.reshape(-1,28*28).astype('float32' )/ 255.0

custom model

In [3]:
class Dense(layers.Layer):
    def __init__(self, units):
        super(Dense, self).__init__()
        self.units = units
        
    
    def build(self, input_shape):
        self.w = self.add_weight(
            name='w',
            shape=(input_shape[-1], self.units),
            initializer = 'random_normal',
            trainable=True,   
        )

        self.b = self.add_weight(
            name='b', shape=(self.units,), initializer='zeros', trainable=True,
        )

    def call (self, inputs):
        return tf.matmul(inputs, self.w) + self.b

In [4]:
class MyRelu(layers.Layer):
    def __init__(self):
        super(MyRelu, self).__init__()

    def call(self, x):
        return tf.math.maximum(x,0)

In [5]:
# class MyModel(keras.Model):
#     def __init__(self, num_classes=10):
#         super(MyModel, self).__init__()
#         self.dense1   = layers.Dense(64)
#         self.dense2   = layers.Dense(num_classes)
    
#     def call(self, input_tensor):
#         x =tf.nn.relu(self.dense1(input_tensor))
#         return self.dense2(x)


class MyModel(keras.Model):
    def __init__(self, num_classes=10):
        super(MyModel, self).__init__()
        self.dense1   = Dense(64)
        self.dense2   = Dense(num_classes)
        self.relu = MyRelu()
    
    def call(self, input_tensor):
        x =self.relu(self.dense1(input_tensor))
        return self.dense2(x)

In [6]:
model = MyModel()

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(),
    metrics=['accuracy']
)

model.fit(x_train, y_train, batch_size=64, epochs=2, verbose=2)
model.evaluate(x_test,y_test, batch_size=64, verbose=2)


Epoch 1/2
938/938 - 3s - 3ms/step - accuracy: 0.8898 - loss: 0.4160
Epoch 2/2
938/938 - 2s - 2ms/step - accuracy: 0.9426 - loss: 0.2009
157/157 - 0s - 3ms/step - accuracy: 0.9515 - loss: 0.1664


[0.16638457775115967, 0.9514999985694885]